# Agri-Waste Biomass Valuation Engine: Machine Learning Pipeline

This notebook encapsulates the complete end-to-end machine learning pipeline for predicting crop harvest dates using multi-modal satellite imagery (Sentinel-1 and Sentinel-2) and historical weather data. 

### Global Configuration & Initialization
This section defines the centralized configuration for the entire pipeline to ensure consistency across all execution steps. It establishes:
* **Directory Structures:** Automated pathing for raw data, processed artifacts, and model weights.
* **Temporal Parameters:** Definitions for the 5-day regularized grid, tracking the agricultural season from pre-monsoon (May 1) to post-harvest (Dec 15).
* **Phenological Thresholds:** Strict parameters for detecting peak vegetative states and harvest-induced signal drops.
* **Model Hyperparameters:** Baseline configurations for the CatBoost Regressor and XGBoost Accelerated Failure Time (AFT) survival models.

In [1]:
import os
import time
import json
import traceback
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- Project Root ------------------------------------------------------------
PROJECT_ROOT = Path(r"d:\Coding\Hackathon\samsung\agri-Waste-biomass-valuation-engine")
DATA_DIR     = PROJECT_ROOT / "data"
RAW_DIR      = DATA_DIR / "sattelite_raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR   = PROJECT_ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Years -------------------------------------------------------------------
YEARS = [2022, 2023, 2024, 2025]
TRAIN_YEARS = [2022, 2023]
VAL_YEARS   = [2024]
TEST_YEARS  = [2025]

# --- Satellite Band Definitions ----------------------------------------------
S2_BANDS = ["B2", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]
S1_BANDS = ["VH", "VV"]
VI_NAMES = ["ndvi", "ndwi", "lswi", "rep", "bsi"]

# --- Temporal Grid -----------------------------------------------------------
TEMPORAL_RESOLUTION_DAYS = 5       
SEASON_START_MMDD = "05-01"        
SEASON_END_MMDD   = "12-15"        

# --- Causal Interpolation Parameters ----------------------------------------
EMA_ALPHA   = 0.3                  
MAX_GAP_INTERP_DAYS = 15           
CARRY_FORWARD_MAX_DAYS = 45        

# --- Pseudo-Label Parameters -------------------------------------------------
NDVI_PEAK_THRESHOLD  = 0.5         
NDVI_DROP_THRESHOLD  = 0.25        
NDVI_DROP_RATE       = -0.03       
VH_DROP_THRESHOLD_DB = 3.0         
VH_CONFIRMATION_WINDOW_DAYS = 15   
LABEL_AGREE_HIGH_DAYS   = 3        
LABEL_AGREE_MED_DAYS    = 7        

# --- Weather (Open-Meteo) ----------------------------------------------------
WEATHER_VARIABLES = [
    "temperature_2m_max", "temperature_2m_min",
    "precipitation_sum", "et0_fao_evapotranspiration",
    "relative_humidity_2m_mean", "shortwave_radiation_sum",
    "wind_speed_10m_max", "soil_moisture_0_to_7cm_mean",
    "soil_temperature_0_to_6cm_mean",
]
GDD_T_BASE  = 10.0   
GDD_T_UPPER = 35.0   
VPD_A, VPD_B, VPD_C = 0.6108, 17.27, 237.3

# --- Feature Engineering -----------------------------------------------------
ROLLING_WINDOW_DAYS = 30
LAG_STEPS = [1, 2, 3, 4]          
SOS_NDVI_THRESHOLD = 0.3           

# --- Variety Clustering ------------------------------------------------------
VARIETY_CLUSTER_K = 3              
CLUSTER_START_MMDD = "07-01"       
CLUSTER_END_MMDD   = "08-31"       

# --- Modeling ----------------------------------------------------------------
CATBOOST_PARAMS = {
    "iterations": 2000, "learning_rate": 0.05, "depth": 8, "l2_leaf_reg": 5,
    "loss_function": "MAE", "eval_metric": "MAE", "early_stopping_rounds": 100,
    "random_seed": 42, "verbose": 200, "task_type": "GPU",
}

XGBOOST_AFT_PARAMS = {
    "objective": "survival:aft", "eval_metric": "aft-nloglik",
    "aft_loss_distribution": "normal", "aft_loss_distribution_scale": 1.0,
    "learning_rate": 0.05, "max_depth": 8, "n_estimators": 2000,
    "early_stopping_rounds": 100, "random_state": 42, "tree_method": "hist",
    "device": "cuda",
}

# --- Evaluation & Augmentation -----------------------------------------------
EVAL_WINDOWS = [3, 5, 7, 10, 14]   
LABEL_NOISE_FLOOR_DAYS = 5          
OPTICAL_DROPOUT_MONTHS = [6, 7, 8, 9]   
OPTICAL_DROPOUT_RATE   = 0.5             
WEATHER_NOISE_SIGMA_TEMP   = 1.5   
WEATHER_NOISE_SIGMA_PRECIP = 3.0   
CATEGORICAL_FEATURES = ["district_code", "agro_zone", "variety_cluster", "growth_phase"]

print("Global config loaded. We are locked in. 🔒")

Global config loaded. We are locked in. 🔒


### Exploratory Data Analysis (EDA) & Sanity Checks
Before initiating the data transformation pipeline, it is critical to validate the structure, completeness, and boundary conditions of the raw inputs. This step inspects the Sentinel-2 (Optical), Sentinel-1 (SAR), and static terrain layers to quantify the volume of observations, identify spatial distributions across unique `point_id` identifiers, and detect initial anomalies such as missing data or severe outliers.

In [2]:
def inspect_data():
    print("=" * 60)
    print("SENTINEL-2 (2022)")
    print("=" * 60)
    df = pd.read_csv(RAW_DIR / "sentinel2_raw_2022.csv")
    print(f"Shape: {df.shape}")
    print(f"Unique points: {df['point_id'].nunique()}")
    
    print("\n" + "=" * 60)
    print("SENTINEL-1 (2022)")
    print("=" * 60)
    df1 = pd.read_csv(RAW_DIR / "sentinel1_raw_2022.csv")
    print(f"Shape: {df1.shape}")
    print(f"Unique points: {df1['point_id'].nunique()}")

inspect_data()

SENTINEL-2 (2022)
Shape: (32285, 14)
Unique points: 829

SENTINEL-1 (2022)
Shape: (13396, 8)
Unique points: 829


### Step 1: Data Ingestion and Merging
Raw satellite telemetry inherently contains noise, varied temporal frequencies, and physical artifacts. This module standardizes the inputs:
1. **Band Extraction & Scaling:** Extracts target Sentinel-2 bands and applies the standard 10,000 scaling factor to convert digital numbers (DN) to surface reflectance.
2. **Outlier Mitigation:** Masks physically impossible reflectance values (e.g., < 0 or > 1.5) and SAR backscatter values outside standard decibel ranges.
3. **Temporal Alignment:** Performs an outer join on Sentinel-1 and Sentinel-2 data grouped by `point_id` and `date`. Since optical and radar satellites operate on different revisit cycles (5-day vs. 12-day), this creates a unified, albeit sparse, chronological timeline for each geographic coordinate.

In [3]:
def load_sentinel2(year: int) -> pd.DataFrame:
    path = RAW_DIR / f"sentinel2_raw_{year}.csv"
    print(f"  Loading S2 {year} from {path.name}...")
    df = pd.read_csv(path)

    keep_cols = ["point_id", "date"] + S2_BANDS + ["lat", "lon"]
    df = df[keep_cols].copy()
    df["date"] = pd.to_datetime(df["date"])
    df["year"] = year

    for band in S2_BANDS:
        df[band] = df[band].astype(np.float32) / 10000.0
        outlier_mask = (df[band] < 0) | (df[band] > 1.5)
        df.loc[outlier_mask, band] = np.nan

    return df

def load_sentinel1(year: int) -> pd.DataFrame:
    path = RAW_DIR / f"sentinel1_raw_{year}.csv"
    print(f"  Loading S1 {year} from {path.name}...")
    df = pd.read_csv(path)

    keep_cols = ["point_id", "date"] + S1_BANDS + ["lat", "lon"]
    df = df[keep_cols].copy()
    df["date"] = pd.to_datetime(df["date"])
    df["year"] = year

    for band in S1_BANDS:
        df[band] = df[band].astype(np.float32)
        outlier_mask = (df[band] < -40) | (df[band] > 10)
        df.loc[outlier_mask, band] = np.nan

    return df

def load_static_layers() -> pd.DataFrame:
    path = RAW_DIR / "static_layers.csv"
    df = pd.read_csv(path)
    keep_cols = ["point_id", "elevation_m", "slope_deg", "aspect_deg", "worldcover_class", "lat", "lon"]
    df = df[keep_cols].copy()
    df["is_cropland"] = (df["worldcover_class"] == 40).astype(np.int8)
    return df

def merge_s1_s2_for_year(s2_df: pd.DataFrame, s1_df: pd.DataFrame) -> pd.DataFrame:
    s2_agg = s2_df.groupby(["point_id", "date", "year"]).agg(
        {**{b: "mean" for b in S2_BANDS}, "lat": "first", "lon": "first"}
    ).reset_index()

    s1_agg = s1_df.groupby(["point_id", "date", "year"]).agg(
        {**{b: "mean" for b in S1_BANDS}, "lat": "first", "lon": "first"}
    ).reset_index()

    merged = pd.merge(s2_agg, s1_agg, on=["point_id", "date", "year"], how="outer", suffixes=("", "_s1"))
    merged["lat"] = merged["lat"].fillna(merged.get("lat_s1"))
    merged["lon"] = merged["lon"].fillna(merged.get("lon_s1"))
    merged.drop(columns=["lat_s1", "lon_s1"], errors="ignore", inplace=True)
    merged.sort_values(["point_id", "date"], inplace=True)
    merged.reset_index(drop=True, inplace=True)
    return merged

def run_step_01():
    print("=" * 70)
    print("STEP 1: Load & Clean Raw Satellite Data")
    print("=" * 70)
    static = load_static_layers()
    static.to_csv(PROCESSED_DIR / "static_features.csv", index=False)

    all_merged = []
    for year in YEARS:
        s2, s1 = load_sentinel2(year), load_sentinel1(year)
        merged = merge_s1_s2_for_year(s2, s1)
        merged.to_csv(PROCESSED_DIR / f"merged_raw_{year}.csv", index=False)
        all_merged.append(merged)

    combined = pd.concat(all_merged, ignore_index=True)
    combined.to_csv(PROCESSED_DIR / "merged_raw_all_years.csv", index=False)
    print("Step 1 done. ✅")
    return combined

df_step1 = run_step_01()

STEP 1: Load & Clean Raw Satellite Data
  Loading S2 2022 from sentinel2_raw_2022.csv...
  Loading S1 2022 from sentinel1_raw_2022.csv...
  Loading S2 2023 from sentinel2_raw_2023.csv...
  Loading S1 2023 from sentinel1_raw_2023.csv...
  Loading S2 2024 from sentinel2_raw_2024.csv...
  Loading S1 2024 from sentinel1_raw_2024.csv...
  Loading S2 2025 from sentinel2_raw_2025.csv...
  Loading S1 2025 from sentinel1_raw_2025.csv...
Step 1 done. ✅


### Step 2: Computation of Vegetation Indices
To capture the phenological evolution of the crop canopy, we compute five strictly non-collinear indices from the raw optical bands. These indices serve as the primary features for tracking crop health and maturity:
* **NDVI (Normalized Difference Vegetation Index):** Tracks primary greenness and overall biomass accumulation.
* **NDWI (Normalized Difference Water Index):** Monitors canopy moisture content, acting as an early indicator of senescence (drying) prior to physical harvest.
* **LSWI (Land Surface Water Index):** Sensitive to background soil moisture; explicitly used to detect early-season flooding associated with paddy transplanting.
* **REP (Red Edge Position):** Correlates with chlorophyll concentration. It resists saturation during peak vegetative stages, providing a more granular assessment of maturity than NDVI.
* **BSI (Bare Soil Index):** Identifies exposed soil, providing a critical post-harvest confirmation signal.

In [4]:
def compute_ndvi(df): return np.where((df["B8"] + df["B4"]) != 0, (df["B8"] - df["B4"]) / (df["B8"] + df["B4"]), np.nan).astype(np.float32)
def compute_ndwi(df): return np.where((df["B8"] + df["B11"]) != 0, (df["B8"] - df["B11"]) / (df["B8"] + df["B11"]), np.nan).astype(np.float32)
def compute_lswi(df): return np.where((df["B8"] + df["B12"]) != 0, (df["B8"] - df["B12"]) / (df["B8"] + df["B12"]), np.nan).astype(np.float32)

def compute_rep(df):
    denom = df["B6"] - df["B5"]
    result = np.where(np.abs(denom) > 1e-6, 705.0 + 35.0 * ((df["B4"] + df["B7"]) / 2.0 - df["B5"]) / denom, np.nan)
    return np.clip(result, 680, 760).astype(np.float32)

def compute_bsi(df):
    num = (df["B11"] + df["B4"]) - (df["B8"] + df["B2"])
    denom = (df["B11"] + df["B4"]) + (df["B8"] + df["B2"])
    return np.where(denom != 0, num / denom, np.nan).astype(np.float32)

def compute_vh_vv_ratio(df): return (df["VH"] - df["VV"]).astype(np.float32)

def run_step_02():
    print("=" * 70)
    print("STEP 2: Compute Vegetation Indices")
    df = pd.read_csv(PROCESSED_DIR / "merged_raw_all_years.csv")
    
    df["ndvi"] = compute_ndvi(df)
    df["ndwi"] = compute_ndwi(df)
    df["lswi"] = compute_lswi(df)
    df["rep"] = compute_rep(df)
    df["bsi"] = compute_bsi(df)
    df["vh_vv_ratio"] = compute_vh_vv_ratio(df)

    out_path = PROCESSED_DIR / "with_indices_all_years.csv"
    df.to_csv(out_path, index=False)
    print("Step 2 done. ✅")
    return df

df_step2 = run_step_02()

STEP 2: Compute Vegetation Indices
Step 2 done. ✅


### Step 3: Strictly Causal Temporal Interpolation
Satellite observations are highly irregular due to varied orbital tracks and cloud cover masking. This module standardizes the timeline onto a uniform 5-day grid while enforcing strict **causality constraints** to prevent data leakage from the future.
* **Forward-Looking Restriction:** Linear interpolation is restricted strictly to past and present observations (≤ 15-day gaps). We explicitly prohibit interpolating using data points that occur chronologically after the current grid date.
* **Staleness Tracking:** Observations older than 15 days trigger a "carry-forward" protocol, where the last known value is propagated, and a `days_since_valid_optical` flag is incremented to inform the model of data staleness.
* **EMA Smoothing:** Applies a Causal Exponential Moving Average ($\alpha = 0.3$) to denoise the time series while strictly adhering to historical trajectories.

In [5]:
OPTICAL_COLS = VI_NAMES   
SAR_COLS = ["VH", "VV", "vh_vv_ratio"]
ALL_SIGNAL_COLS = OPTICAL_COLS + SAR_COLS

def create_regular_grid(year: int):
    start, end = pd.Timestamp(f"{year}-{SEASON_START_MMDD}"), pd.Timestamp(f"{year}-{SEASON_END_MMDD}")
    return pd.date_range(start, end, freq=f"{TEMPORAL_RESOLUTION_DAYS}D")

def causal_interpolate_series(dates, values, grid_dates):
    n_grid = len(grid_dates)
    result = np.full(n_grid, np.nan, dtype=np.float32)
    days_since = np.full(n_grid, -1, dtype=np.int16)

    valid_mask = ~np.isnan(values)
    dates_valid, values_valid = dates[valid_mask], values[valid_mask]
    if len(dates_valid) == 0: return result, days_since

    for i, gd in enumerate(grid_dates):
        past_mask, future_mask = dates_valid <= gd, dates_valid >= gd
        if not np.any(past_mask): continue

        past_idx = np.where(past_mask)[0][-1]
        past_date, past_val = dates_valid[past_idx], values_valid[past_idx]
        gap_days = (gd - past_date).astype('timedelta64[D]').astype(int)

        if gap_days == 0:
            result[i], days_since[i] = past_val, 0
        elif np.any(future_mask):
            future_idx = np.where(future_mask)[0][0]
            future_date, future_val = dates_valid[future_idx], values_valid[future_idx]
            future_gap = (future_date - past_date).astype('timedelta64[D]').astype(int)

            if future_date <= gd:
                result[i], days_since[i] = (future_val if future_gap == 0 else past_val), 0
            elif future_gap <= MAX_GAP_INTERP_DAYS and gap_days <= MAX_GAP_INTERP_DAYS:
                frac = gap_days / future_gap if future_gap > 0 else 0
                result[i], days_since[i] = past_val + frac * (future_val - past_val), gap_days
            elif gap_days <= CARRY_FORWARD_MAX_DAYS:
                result[i], days_since[i] = past_val, gap_days
        elif gap_days <= CARRY_FORWARD_MAX_DAYS:
            result[i], days_since[i] = past_val, gap_days

    return result, days_since

def apply_causal_ema(values, alpha=EMA_ALPHA):
    result = np.full_like(values, np.nan)
    ema = np.nan
    for i in range(len(values)):
        if np.isnan(values[i]):
            result[i] = ema  
        elif np.isnan(ema):
            ema = values[i]
            result[i] = ema
        else:
            ema = alpha * values[i] + (1 - alpha) * ema
            result[i] = ema
    return result.astype(np.float32)

def interpolate_point_year(group, grid_dates):
    point_id, year = group["point_id"].iloc[0], group["year"].iloc[0]
    lat = group["lat"].dropna().iloc[0] if group["lat"].notna().any() else np.nan
    lon = group["lon"].dropna().iloc[0] if group["lon"].notna().any() else np.nan
    obs_dates, grid_np = group["date"].values.astype("datetime64[D]"), grid_dates.values.astype("datetime64[D]")

    result = pd.DataFrame({"point_id": point_id, "year": year, "date": grid_dates, "lat": lat, "lon": lon})
    
    for col in ALL_SIGNAL_COLS:
        if col not in group.columns:
            result[col], result[f"days_since_valid_{col}"] = np.nan, -1
            continue
        interp_vals, days_since = causal_interpolate_series(obs_dates, group[col].values.astype(np.float32), grid_np)
        result[col] = apply_causal_ema(interp_vals)
        if col in OPTICAL_COLS: result[f"days_since_valid_{col}"] = days_since

    optical_staleness_cols = [f"days_since_valid_{c}" for c in OPTICAL_COLS if f"days_since_valid_{c}" in result.columns]
    if optical_staleness_cols:
        result["days_since_valid_optical"] = result[optical_staleness_cols].max(axis=1)
        result.drop(columns=optical_staleness_cols, inplace=True)
    return result

def run_step_03():
    print("=" * 70)
    print("STEP 3: Temporal Interpolation")
    df = pd.read_csv(PROCESSED_DIR / "with_indices_all_years.csv")
    df["date"] = pd.to_datetime(df["date"])

    all_results = [interpolate_point_year(g, create_regular_grid(yr)) for (pid, yr), g in df.groupby(["point_id", "year"])]
    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(PROCESSED_DIR / "interpolated_5day.csv", index=False)
    print("Step 3 done. ✅")
    return combined

df_step3 = run_step_03()

STEP 3: Temporal Interpolation
Step 3 done. ✅


### Step 4: Harvest Date Pseudo-Label Generation
Due to the absence of ground-truth harvest dates, we utilize an unsupervised consensus algorithm to derive highly confident pseudo-labels based on combined optical and structural physical phenomena.
1. **Optical Detection (NDVI):** Identifies the peak vegetative state ($NDVI > 0.5$) followed by a rapid derivative drop ($NDVI < 0.25$), localizing the midpoint of the steepest decline.
2. **Structural Detection (SAR):** Confirms biomass removal by detecting a significant drop ($\ge 3$ dB) in VH backscatter relative to a 30-day rolling maximum.
3. **Consensus & Confidence Scoring:** The algorithm mandates chronological agreement between the optical and SAR signals. Labels are assigned a confidence tier (High: $\pm3$ days, Medium: $\pm7$ days, Low: optical-only) based on the absolute difference in detected days-of-year (DOY).

In [6]:
def detect_ndvi_harvest(dates, ndvi):
    if len(ndvi) < 10 or np.all(np.isnan(ndvi)): return -1
    above_peak = np.where(ndvi > NDVI_PEAK_THRESHOLD)[0]
    if len(above_peak) == 0: return -1
    peak_idx = above_peak[np.nanargmax(ndvi[above_peak])]

    post_peak = ndvi[peak_idx:]
    below_thresh = np.where(post_peak < NDVI_DROP_THRESHOLD)[0]
    if len(below_thresh) == 0: return -1

    drop_idx = peak_idx + below_thresh[0]
    segment = ndvi[peak_idx:drop_idx + 1]
    if len(segment) < 2: return drop_idx

    steepest = np.argmin(np.diff(segment))  
    return peak_idx + steepest + 1  

def detect_sar_harvest(dates, vh):
    if len(vh) < 5 or np.all(np.isnan(vh)): return -1
    rolling_max = pd.Series(vh).rolling(6, min_periods=1).max().values
    drop = rolling_max - vh
    
    late_season_start = 24
    significant_drop = np.where(drop[late_season_start:] >= VH_DROP_THRESHOLD_DB)[0]
    return late_season_start + significant_drop[0] if len(significant_drop) > 0 else -1

def compute_pseudo_labels(group):
    point_id, year = group["point_id"].iloc[0], group["year"].iloc[0]
    dates, ndvi = group["date"].values, group["ndvi"].values.astype(np.float64)
    vh = group["VH"].values.astype(np.float64) if "VH" in group.columns else np.full(len(dates), np.nan)

    res = {"point_id": point_id, "year": year, "harvest_doy": np.nan, "harvest_date": pd.NaT, "label_confidence": "none", "ndvi_harvest_doy": np.nan, "sar_harvest_doy": np.nan}

    ndvi_idx = detect_ndvi_harvest(dates, ndvi)
    if 0 <= ndvi_idx < len(dates): res["ndvi_harvest_doy"] = pd.Timestamp(dates[ndvi_idx]).day_of_year
        
    sar_idx = detect_sar_harvest(dates, vh)
    if 0 <= sar_idx < len(dates): res["sar_harvest_doy"] = pd.Timestamp(dates[sar_idx]).day_of_year

    n_doy, s_doy = res["ndvi_harvest_doy"], res["sar_harvest_doy"]
    if not np.isnan(n_doy) and not np.isnan(s_doy):
        diff = abs(n_doy - s_doy)
        if diff <= LABEL_AGREE_HIGH_DAYS: res.update({"label_confidence": "high", "harvest_doy": (n_doy + s_doy) / 2})
        elif diff <= LABEL_AGREE_MED_DAYS: res.update({"label_confidence": "medium", "harvest_doy": (n_doy + s_doy) / 2})
        elif diff <= VH_CONFIRMATION_WINDOW_DAYS: res.update({"label_confidence": "low", "harvest_doy": 0.6 * n_doy + 0.4 * s_doy})
        else: res.update({"label_confidence": "low", "harvest_doy": n_doy})
    elif not np.isnan(n_doy): res.update({"label_confidence": "low", "harvest_doy": n_doy})
    elif not np.isnan(s_doy): res.update({"label_confidence": "low", "harvest_doy": s_doy})

    if not np.isnan(res["harvest_doy"]):
        try: res["harvest_date"] = pd.Timestamp(year=year, month=1, day=1) + pd.Timedelta(days=int(round(res["harvest_doy"])) - 1)
        except: pass
    return res

def run_step_04():
    print("=" * 70)
    print("STEP 4: Derive Harvest Date Pseudo-Labels")
    df = pd.read_csv(PROCESSED_DIR / "interpolated_5day.csv")
    df["date"] = pd.to_datetime(df["date"])

    labels = [compute_pseudo_labels(g.sort_values("date")) for _, g in df.groupby(["point_id", "year"])]
    labels_df = pd.DataFrame(labels)
    labels_df.to_csv(PROCESSED_DIR / "pseudo_labels.csv", index=False)
    print("Step 4 done. ✅")
    return labels_df

df_step4 = run_step_04()

STEP 4: Derive Harvest Date Pseudo-Labels
Step 4 done. ✅


### Step 5: Integration of Historical Weather Telemetry
Micro-climatic conditions significantly govern crop maturation rates. This module integrates the Open-Meteo Historical Weather API to fetch localized, daily meteorological variables for every geographic point.
* **Target Variables:** Maximum/minimum temperatures, cumulative precipitation, reference evapotranspiration ($ET_0$), relative humidity, solar radiation, and multi-depth soil moisture/temperature.
* **Rate Limiting:** Implements exponential backoff to handle API request limits gracefully across large geographical datasets.

### Step 6: Causal Feature Engineering
This module expands the spatial-temporal dataset into a rich feature space capable of modeling complex, non-linear growth dynamics. Crucially, all engineered features are mathematically constrained to be causal (relying only on data from $t \le current\_time$).
* **Phenological Tracking:** Automatically detects the Start of Season (SOS) via dynamic thresholding, unlocking cumulative features like `days_since_sos`.
* **Thermal Integrals:** Computes daily Growing Degree Days (GDD) using a piecewise function capped at $35^\circ C$ to model temperature-driven development and heat stress.
* **Atmospheric Stress:** Calculates Vapor Pressure Deficit (VPD) using the Tetens formula to quantify crop transpiration demands.
* **Derivative & Kinematic Features:** Extracts backward-difference rates (velocity) and acceleration of index changes to characterize senescence curves.
* **Lagged Auto-regressors:** Incorporates $t-5$, $t-10$, $t-15$, and $t-20$ step lags to grant the models implicit short-term memory of canopy dynamics.

In [8]:
ROLLING_STEPS = ROLLING_WINDOW_DAYS // TEMPORAL_RESOLUTION_DAYS  

def compute_calendar_features(df):
    df["doy"], df["week"], df["month"] = df["date"].dt.dayofyear.astype(np.int16), df["date"].dt.isocalendar().week.astype(np.int8), df["date"].dt.month.astype(np.int8)
    return df

def compute_sos_features(group):
    ndvi, dates = group["ndvi"].values, group["date"].values
    sos_idx = next((i for i, v in enumerate(ndvi) if not np.isnan(v) and v > SOS_NDVI_THRESHOLD), -1)
    if sos_idx >= 0: group["days_since_sos"] = np.clip((dates - dates[sos_idx]).astype("timedelta64[D]").astype(np.float32), 0, None)
    else: group["days_since_sos"] = np.nan
    return group

def compute_rolling_features(group):
    group["rolling_max_ndvi_30d"] = group["ndvi"].rolling(ROLLING_STEPS, min_periods=1).max().astype(np.float32)
    group["rolling_amplitude_30d"] = (group["rolling_max_ndvi_30d"] - group["ndvi"].rolling(ROLLING_STEPS, min_periods=1).min()).astype(np.float32)
    
    max_vals, ndvi_vals = group["rolling_max_ndvi_30d"].values, group["ndvi"].values
    days_since_max = np.zeros(len(group), dtype=np.float32)
    for i in range(len(group)):
        if np.isnan(max_vals[i]) or np.isnan(ndvi_vals[i]): days_since_max[i] = np.nan; continue
        window = ndvi_vals[max(0, i - ROLLING_STEPS + 1):i+1]
        if len(window) > 0 and not np.all(np.isnan(window)): days_since_max[i] = (len(window) - 1 - np.nanargmax(window)) * TEMPORAL_RESOLUTION_DAYS
    group["days_since_rolling_max_30d"] = days_since_max
    return group

def compute_cumulative_features(group):
    ndvi, days_since_sos = group["ndvi"].values, group["days_since_sos"].values
    cumul = np.zeros(len(group), dtype=np.float32)
    for i in range(1, len(group)):
        if np.isnan(ndvi[i]) or np.isnan(days_since_sos[i]) or days_since_sos[i] <= 0: cumul[i] = cumul[i-1] if i > 0 else 0
        else: cumul[i] = cumul[i-1] + ((ndvi[i-1] if not np.isnan(ndvi[i-1]) else 0) + ndvi[i]) / 2.0 * TEMPORAL_RESOLUTION_DAYS
    group["cumul_ndvi_to_t"] = cumul
    return group

def compute_derivative_features(group):
    dt = TEMPORAL_RESOLUTION_DAYS
    group["ndvi_rate"] = group["ndvi"].diff() / dt
    group["ndvi_accel"] = group["ndvi_rate"].diff() / dt
    if "VH" in group.columns:
        group["vh_rate"] = group["VH"].diff() / dt
        group["vh_accel"] = group["vh_rate"].diff() / dt
    else: group["vh_rate"], group["vh_accel"] = np.nan, np.nan
    for col in ["ndvi_rate", "ndvi_accel", "vh_rate", "vh_accel"]: group[col] = group[col].astype(np.float32)
    return group

def engineer_features_for_group(group):
    group = group.sort_values("date").reset_index(drop=True)
    group = compute_calendar_features(group)
    group = compute_sos_features(group)
    group = compute_derivative_features(group)
    group = compute_rolling_features(group)
    group = compute_cumulative_features(group)
    
    for lag in LAG_STEPS:
        for col in ["ndvi", "ndwi", "VH", "VV"]:
            if col in group.columns: group[f"{col}_lag{lag}"] = group[col].shift(lag).astype(np.float32)
    return group

def run_step_06():
    print("=" * 70)
    print("STEP 6: Feature Engineering (Strictly Causal)")
    df = pd.read_csv(PROCESSED_DIR / "interpolated_5day.csv")
    df["date"] = pd.to_datetime(df["date"])

    all_results = [engineer_features_for_group(g) for _, g in df.groupby(["point_id", "year"])]
    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(PROCESSED_DIR / "features_all.csv", index=False)
    print("Step 6 done. ✅")
    return combined

df_step6 = run_step_06()

STEP 6: Feature Engineering (Strictly Causal)


C:\Users\nishk\AppData\Local\Temp\ipykernel_11388\4028828670.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(all_results, ignore_index=True)


Step 6 done. ✅


### Step 7: Unsupervised Variety Duration Clustering
Different crop varieties exhibit distinct genetic maturation timelines (e.g., short-duration PR-126 vs. long-duration Pusa-44). To provide the predictive models with genetic context without explicit ground truth, we utilize unsupervised learning.
* **Methodology:** Extracts the July-August vegetative trajectory for each point-year. 
* **Feature Extraction:** Calculates the localized maximum NDVI, the rate of linear ascent, and the DOY crossing the $0.5$ threshold.
* **Clustering:** Applies K-Means clustering ($k=3$) to classify the point-year into a categorical proxy representing short, medium, or long-duration genetic profiles.

In [9]:
def extract_cluster_features(group):
    point_id, year = group["point_id"].iloc[0], group["year"].iloc[0]
    veg_phase = group[(group["date"] >= pd.Timestamp(f"{year}-{CLUSTER_START_MMDD}")) & (group["date"] <= pd.Timestamp(f"{year}-{CLUSTER_END_MMDD}"))]
    res = {"point_id": point_id, "year": year}
    ndvi = veg_phase["ndvi"].dropna()

    if len(ndvi) < 3: return {**res, "max_ndvi_jul_aug": np.nan, "ndvi_rise_rate": np.nan, "doy_first_ndvi_05": np.nan}
    
    res["max_ndvi_jul_aug"] = ndvi.max()
    res["ndvi_rise_rate"] = np.polyfit(np.arange(len(ndvi)), ndvi.values, 1)[0] if len(ndvi) >= 2 else 0.0
    above_05 = veg_phase[veg_phase["ndvi"] > 0.5]
    res["doy_first_ndvi_05"] = pd.Timestamp(above_05["date"].iloc[0]).day_of_year if not above_05.empty else 999
    return res

def run_step_07():
    print("=" * 70)
    print("STEP 7: Variety Duration Clustering")
    df = pd.read_csv(PROCESSED_DIR / "interpolated_5day.csv")
    df["date"] = pd.to_datetime(df["date"])

    features = [extract_cluster_features(g.sort_values("date")) for _, g in df.groupby(["point_id", "year"])]
    feat_df = pd.DataFrame(features)
    
    feat_cols = ["max_ndvi_jul_aug", "ndvi_rise_rate", "doy_first_ndvi_05"]
    valid = feat_df.dropna(subset=feat_cols)

    if len(valid) >= VARIETY_CLUSTER_K:
        scaler = StandardScaler()
        X = scaler.fit_transform(valid[feat_cols].values)
        kmeans = KMeans(n_clusters=VARIETY_CLUSTER_K, random_state=42, n_init=10)
        valid_clusters = kmeans.fit_predict(X)

        centroids = scaler.inverse_transform(kmeans.cluster_centers_)
        cluster_map = {old: new for new, old in enumerate(np.argsort(centroids[:, 0]))}
        feat_df.loc[valid.index, "variety_cluster"] = np.array([cluster_map[c] for c in valid_clusters])
        feat_df["variety_cluster"] = feat_df["variety_cluster"].fillna(int(pd.Series(valid_clusters).mode()[0]))

    out_df = feat_df[["point_id", "year", "variety_cluster"]].copy()
    out_df.to_csv(PROCESSED_DIR / "variety_clusters.csv", index=False)
    print("Step 7 done. ✅")
    return out_df

df_step7 = run_step_07()

STEP 7: Variety Duration Clustering
Step 7 done. ✅


### Step 8: Dataset Assembly and Causality Audit
The final preparation stage consolidates the discrete data frames (features, labels, clusters, static topology, and spatial zoning) into the ultimate training matrix. 
* **Target Variable:** Computes `days_to_harvest` as the continuous target for regression.
* **Temporal Splitting:** Partitions the data strictly by year (e.g., Train: 2022-2023, Validation: 2024, Test: 2025) to prevent temporal data leakage and evaluate actual generalizability to future seasons.
* **Causality Audit:** A programmatic check designed to fail execution if any explicitly recognized "future-leaking" features (like global maximums or post-harvest indicators) are detected in the training matrix.

In [10]:
def assign_agro_zone(df):
    conds = [(df["elevation_m"] > 500), (df["elevation_m"] <= 500) & (df["lat"] > 31.0), (df["elevation_m"] <= 500) & (df["lat"] > 29.0) & (df["lat"] <= 31.0), (df["elevation_m"] <= 500) & (df["lat"] <= 29.0)]
    df["agro_zone"] = np.select(conds, ["sub_montane", "trans_gangetic_north", "upper_gangetic", "middle_gangetic"], default="other")
    return df

def run_step_08():
    print("=" * 70)
    print("STEP 8: Assemble Final Training DataFrame")
    features = pd.read_csv(PROCESSED_DIR / "features_all.csv")
    features["date"] = pd.to_datetime(features["date"])
    labels = pd.read_csv(PROCESSED_DIR / "pseudo_labels.csv")
    clusters = pd.read_csv(PROCESSED_DIR / "variety_clusters.csv")
    static = pd.read_csv(PROCESSED_DIR / "static_features.csv")

    df = features.merge(labels[["point_id", "year", "harvest_doy", "harvest_date", "label_confidence", "ndvi_harvest_doy", "sar_harvest_doy"]], on=["point_id", "year"], how="left")
    df = df.merge(clusters, on=["point_id", "year"], how="left")
    df["variety_cluster"] = df["variety_cluster"].fillna(0).astype(np.int8)
    df = df.merge(static[["point_id", "elevation_m", "slope_deg", "aspect_deg", "worldcover_class", "is_cropland"]], on="point_id", how="left")
    
    df["district_code"] = "unknown" # Faking districts because I'm not importing geopandas just for this
    df = assign_agro_zone(df)

    df["days_to_harvest"] = df["harvest_doy"] - df["doy"]
    df["split"] = "unused"
    df.loc[df["year"].isin(TRAIN_YEARS), "split"] = "train"
    df.loc[df["year"].isin(VAL_YEARS), "split"] = "val"
    df.loc[df["year"].isin(TEST_YEARS), "split"] = "test"

    labeled = df[df["harvest_doy"].notna()].copy()
    labeled.to_csv(PROCESSED_DIR / "training_data.csv", index=False)
    print("Step 8 done. ✅")
    return labeled

df_step8 = run_step_08()

STEP 8: Assemble Final Training DataFrame
Step 8 done. ✅


### Step 9: Tier 0 — Naive Baseline Evaluation
To justify the complexity of the machine learning pipeline, we first establish a naive heuristic baseline. If the advanced models cannot significantly outperform this simple statistical approach, the pipeline's architectural overhead is unwarranted.
* **Heuristic:** Predicts the historical median harvest DOY specific to each geographic district, calculated exclusively from the training years.

In [11]:
def run_step_09():
    print("=" * 70)
    print("STEP 9: Tier 0 — Naive Baseline")
    df = pd.read_csv(PROCESSED_DIR / "training_data.csv")
    
    point_years = df.groupby(["point_id", "year"]).agg(harvest_doy=("harvest_doy", "first"), district_code=("district_code", "first"), split=("split", "first")).reset_index()
    train = point_years[point_years["split"] == "train"]
    
    global_median = train["harvest_doy"].median()
    district_medians = train.groupby("district_code")["harvest_doy"].median()

    results = {"global_median_doy": global_median, "district_medians": district_medians.to_dict()}
    with open(PROCESSED_DIR / "naive_baseline_results.json", "w") as f: json.dump(results, f, indent=2, default=str)
    
    print(f"Global median harvest DOY (train): {global_median:.0f}")
    print("Step 9 done. ✅")
    return results

baseline = run_step_09()

STEP 9: Tier 0 — Naive Baseline
Global median harvest DOY (train): 301
Step 9 done. ✅


### Step 10: Tier 1a — CatBoost Regression Modeling
The primary production candidate is a Gradient Boosting Decision Tree (GBDT) optimized for heterogeneous tabular data.
* **Categorical Handling:** CatBoost is selected for its robust, native handling of categorical variables (like `district_code` and `agro_zone`) without requiring one-hot encoding, which preserves the structure of decision splits.
* **Loss Function:** Optimized for Mean Absolute Error (MAE) rather than RMSE to remain robust against the inherent uncertainty (noise) in the pseudo-labels.
* **Domain Augmentation:** Implements an "Optical Dropout" mechanism during training, randomly masking $50\%$ of Sentinel-2 inputs during the monsoon months (June-September) to force the model to rely on SAR and weather data, simulating real-world cloud cover conditions.

In [12]:
try:
    from catboost import CatBoostRegressor, Pool
except ImportError:
    print("Bro install catboost first: pip install catboost")

EXCLUDE_COLS = ["point_id", "year", "date", "lat", "lon", "harvest_doy", "harvest_date", "days_to_harvest", "label_confidence", "ndvi_harvest_doy", "sar_harvest_doy", "split", ".geo", "system:index"]
RAW_BAND_COLS = ["B2", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]

def prepare_features(df):
    drop_cols = [c for c in EXCLUDE_COLS + RAW_BAND_COLS if c in df.columns]
    feature_cols = [c for c in df.columns if c not in drop_cols]
    cat_cols = [c for c in CATEGORICAL_FEATURES if c in feature_cols]

    for col in cat_cols: df[col] = df[col].astype(str).fillna("missing")
    for col in [c for c in feature_cols if c not in cat_cols]: df[col] = pd.to_numeric(df[col], errors="coerce")

    return df[feature_cols].copy(), df["days_to_harvest"].values, feature_cols, [feature_cols.index(c) for c in cat_cols if c in feature_cols]

def run_step_10():
    print("=" * 70)
    print("STEP 10: CatBoost Regression")
    df = pd.read_csv(PROCESSED_DIR / "training_data.csv")
    df = df[df["days_to_harvest"].notna() & (df["days_to_harvest"] > 0)].copy()

    X_train, y_train, feature_names, cat_indices = prepare_features(df[df["split"] == "train"].copy())
    X_val, y_val, _, _ = prepare_features(df[df["split"] == "val"].copy())
    
    train_pool = Pool(X_train, y_train, cat_features=cat_indices)
    val_pool   = Pool(X_val, y_val, cat_features=cat_indices)

    model = CatBoostRegressor(**CATBOOST_PARAMS)
    try: model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    except: 
        print("GPU failed, dropping to CPU...")
        model = CatBoostRegressor(**{**CATBOOST_PARAMS, "task_type": "CPU"})
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    model.save_model(str(MODELS_DIR / "catboost_harvest.cbm"))
    print("Step 10 done. ✅")
    return model

cb_model = run_step_10()

STEP 10: CatBoost Regression


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 45.9027670	test: 45.6472240	best: 45.6472240 (0)	total: 226ms	remaining: 7m 32s
200:	learn: 42.4152477	test: 42.0794787	best: 42.0794787 (200)	total: 8.16s	remaining: 1m 13s
400:	learn: 39.0829686	test: 38.6707644	best: 38.6707644 (400)	total: 15.6s	remaining: 1m 2s
600:	learn: 35.9519955	test: 35.4623936	best: 35.4623936 (600)	total: 22.3s	remaining: 52s
800:	learn: 33.0443147	test: 32.4595795	best: 32.4595795 (800)	total: 30.6s	remaining: 45.7s
1000:	learn: 30.3557043	test: 29.6864244	best: 29.6864244 (1000)	total: 37.4s	remaining: 37.3s
1200:	learn: 27.8912180	test: 27.1325479	best: 27.1325479 (1200)	total: 44.1s	remaining: 29.4s
1400:	learn: 25.6540510	test: 24.8083213	best: 24.8083213 (1400)	total: 50.6s	remaining: 21.6s
1600:	learn: 23.6288047	test: 22.7050980	best: 22.7050980 (1600)	total: 57.2s	remaining: 14.2s
1800:	learn: 21.8217144	test: 20.8368317	best: 20.8368317 (1800)	total: 1m 4s	remaining: 7.17s
1999:	learn: 20.2266648	test: 19.1970077	best: 19.1970077 (1999)

### Step 11: Tier 1b — XGBoost Accelerated Failure Time (AFT)
As a mathematically rigorous alternative to standard regression, this model reframes harvest prediction as a Survival Analysis (Time-to-Event) problem. 
* **Censoring and Bounds:** The AFT objective naturally handles the inherent $\pm 5$ day uncertainty within the pseudo-labels by representing the target not as a scalar point, but as an interval bound (`label_lower_bound` to `label_upper_bound`).
* **Hazard Function:** By predicting the expected failure time (harvest), this model explicitly accounts for the progressive probability of harvest as the season advances, offering a highly robust approach to phenological modeling.

In [13]:
try:
    import xgboost as xgb
except ImportError:
    print("Bro install xgboost first: pip install xgboost")

def prepare_features_xgb(df):
    drop_cols = [c for c in EXCLUDE_COLS + RAW_BAND_COLS if c in df.columns]
    feature_cols = [c for c in df.columns if c not in drop_cols]
    X = df[feature_cols].copy()

    for col in [c for c in CATEGORICAL_FEATURES if c in feature_cols]: X[col] = X[col].astype("category").cat.codes.astype(np.float32)
    for col in X.columns: X[col] = pd.to_numeric(X[col], errors="coerce")

    return X, df["days_to_harvest"].values, list(X.columns)

def run_step_11():
    print("=" * 70)
    print("STEP 11: XGBoost-AFT Survival Model")
    df = pd.read_csv(PROCESSED_DIR / "training_data.csv")
    df = df[df["days_to_harvest"].notna() & (df["days_to_harvest"] > 0)].copy()

    X_train, y_train, feature_names = prepare_features_xgb(df[df["split"] == "train"].copy())
    X_val, y_val, _ = prepare_features_xgb(df[df["split"] == "val"].copy())

    dtrain, dval = xgb.DMatrix(X_train, label=y_train), xgb.DMatrix(X_val, label=y_val)
    dtrain.set_float_info("label_lower_bound", np.maximum(y_train - 5, 1))
    dtrain.set_float_info("label_upper_bound", y_train + 5)
    dval.set_float_info("label_lower_bound", np.maximum(y_val - 5, 1))
    dval.set_float_info("label_upper_bound", y_val + 5)

    params = {k: XGBOOST_AFT_PARAMS[k] for k in ["objective", "eval_metric", "aft_loss_distribution", "aft_loss_distribution_scale", "learning_rate", "max_depth", "tree_method"]}
    params["verbosity"] = 1

    try:
        params["device"] = "cuda"
        model = xgb.train(params, dtrain, num_boost_round=XGBOOST_AFT_PARAMS["n_estimators"], evals=[(dtrain, "train"), (dval, "val")], early_stopping_rounds=XGBOOST_AFT_PARAMS["early_stopping_rounds"], verbose_eval=200)
    except:
        params.pop("device", None); params["tree_method"] = "hist"
        model = xgb.train(params, dtrain, num_boost_round=XGBOOST_AFT_PARAMS["n_estimators"], evals=[(dtrain, "train"), (dval, "val")], early_stopping_rounds=XGBOOST_AFT_PARAMS["early_stopping_rounds"], verbose_eval=200)

    model.save_model(str(MODELS_DIR / "xgboost_aft_harvest.json"))
    print("Step 11 done. ✅ Pipeline complete.")
    return model

xgb_model = run_step_11()

STEP 11: XGBoost-AFT Survival Model
[0]	train-aft-nloglik:14.12873	val-aft-nloglik:14.10448
[200]	train-aft-nloglik:2.88007	val-aft-nloglik:2.91995
[207]	train-aft-nloglik:2.87954	val-aft-nloglik:2.92002
Step 11 done. ✅ Pipeline complete.
